# B2-021 — Session 1: Image Patches and Vision Transformers

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260901`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, model hubs, pretrained models, checkpoints, or student data.


## 1. Images become non-overlapping patch tokens

Let images be float32 `(B,C,H,W)=(2,1,4,4)` and patch side `P=2`. Require `H%P=W%P=0`. Reshape to `(B,C,H/P,P,W/P,P)`, permute to row-major patch-grid order `(B,H/P,W/P,C,P,P)`, make contiguous, then reshape to `(B,N,C*P*P)` with `N=(H/P)(W/P)=4`. Token 0 is the top-left patch, then top-right, bottom-left, bottom-right. `reshape`, `permute`, and `contiguous` are taught here; no hidden image library is needed.

**Checkpoint 1A.** What is the patch-token shape?

**Checkpoint 1B.** Why call `contiguous()` after `permute`?

**Collected answers.** The shape is `(2,4,4)`; permutation changes strides, so contiguous storage makes the final reshape's ordering explicit.


## 2. Linear patch projection

A learned `nn.Linear(C*P*P,D)` with `D=8` maps each four-value patch to an eight-value token independently: `(B,4,4)->(B,4,8)`. This is a matrix multiply on the last axis, not a convolution over the token axis. Its weight shape is `(8,4)` because Torch stores linear weights as `(out_features,in_features)`.

**Checkpoint 2A.** Which dimension is contracted?

**Checkpoint 2B.** What is the projected shape?

**Collected answers.** The flattened patch dimension 4 is contracted; the result is `(B,4,8)`.


## 3. Class token and learned positions

Store one float32 parameter `cls_token` shape `(1,1,8)`, expand it to `(B,1,8)`, and concatenate on token axis 1. Store learned positions shape `(1,N+1,8)` and add them elementwise. The class token is a trainable summary slot, patch tokens carry image content, and position vectors label order without adding tokens. The encoder input is `(B,5,8)`.

**Checkpoint 3A.** Does the position table create a sixth token?

**Checkpoint 3B.** Which token is read by the head?

**Collected answers.** No, it is added to five tokens; read encoder output `x[:,0,:]`.


## 4. Worked patchification-to-logit shape ledger

For one `(1,1,4,4)` image with `P=2,D=8`: image `(1,1,4,4)` → four row-major patches `(1,4,4)` → linear projection `(1,4,8)` → prepend class `(1,5,8)` → add positions `(1,5,8)` → one-head encoder `(1,5,8)` → select class `(1,8)` → `Linear(8,2)` logits `(1,2)`. Cross-entropy receives raw float32 logits and int64 target `(1,)`; softmax is for interpretation, not before the loss.

**Checkpoint 4A.** What is `N+1`?

**Checkpoint 4B.** Which target dtype does cross-entropy require?

**Collected answers.** `N+1=5`; the class target is `torch.int64`.


## 5. ViT versus CNN

A CNN hard-codes locality and shared kernels, usually helping with limited data and compute. A ViT exposes global token interactions but self-attention costs `O((N+1)^2 D)` score work and has weaker built-in locality. Larger patches reduce `N` and attention cost but discard fine detail. Compare models at matched data, parameter budget, and resolution rather than declaring a universal winner.

**Checkpoint 5A.** Which model has the stronger locality bias?

**Checkpoint 5B.** What happens to score cost if each image side doubles at fixed patch size?

**Collected answers.** The CNN; `N` quadruples, so quadratic score work grows by about sixteen times.


## 6. Fixed-seed tiny classification train/eval trace

Here is a recorded reference-run lesson-only mini-trace computed from four literal 4x4 bar images, two distinct held-out bar images, patch side 2, width 4, one head, seed `20260901`, and 60 AdamW updates at `lr=.03`. It does not import `vision_fixture.py` and is not p17's model or answer.

| update | train CE | held-out CE | held-out accuracy | class shape |
|---:|---:|---:|---:|---|
| 0 | 0.697418 | 0.698795 | 0.50 | `(2,1,4)` |
| 1 | 0.694295 | 0.696090 | 0.50 | `(2,1,4)` |
| 30 | 0.000007 | 0.000179 | 1.00 | `(2,1,4)` |
| 60 | 0.000000 | 0.000000 | 1.00 | `(2,1,4)` |

The baseline is update 0 before any `step`. The intermediate row shows that one small change is not a learning certificate; the final comparison uses held-out loss and accuracy, not train loss alone. Only the four training rows entered `backward`; both held-out rows were evaluated under `no_grad`. p17 uses different data, width, update count, and thresholds, so these numbers are not reference answers.

**Checkpoint 6A.** How do the lesson demonstration and p17 update-count/lr protocols differ?

**Checkpoint 6B.** May held-out rows enter `backward()`?

**Collected answers.** The lesson-only demonstration uses exactly 60 updates at `lr=.03`; p17 later requires exactly 12 updates at `lr=.05`. Held-out rows never enter `backward()`.


## 7. Worked laboratory: trace values, not only shapes

Use the first code cell below as a transparent patchification microscope. Before running it, copy the sixteen image values onto a 4x4 grid and circle the four 2x2 regions. The reshape introduces separate patch-grid and within-patch axes; the permutation moves the patch-grid axes before channel and within-patch axes. That is why `contiguous()` is not decorative. Predict all four tokens, then compare the exact matrix. A wrong middle-token order can preserve shape and even preserve the set of values, so a shape assertion alone cannot certify spatial semantics.

Next trace the class token and position table separately. Projection changes the last axis from four pixel values to eight learned features. Prepending changes token count from four to five. Adding positions changes neither count nor width. Use an asymmetric position table: if every position row were zero or identical, a broken implementation that ignored positions could pass the probe. Likewise, use a nonidentity projection so returning raw patches padded with zeros cannot pass. The p07 witness deliberately checks the whole output, not one class row.

For the encoder, name four tensors: `tokens`, `positioned`, `attended`, and `encoded`. The exact path is `positioned=tokens+positions`, self-attention with positioned Q/K/V, `residual=positioned+attended`, then `encoded=LayerNorm(residual)`. Normalizing before the residual is a different architecture. Selecting token zero before attention is also different because the class token would never receive patch information. The second code cell prints a shape ledger and checks that LayerNorm leaves each token approximately zero-mean and unit-variance across its eight features.

Now do an error analysis. If projected patches have shape `(B,8,4)`, the linear layer was applied or transposed on the wrong axis. If concatenation reports a size mismatch, check that the expanded class parameter has batch size `B` but remains one token. If logits are nearly identical for every image, inspect whether positions, patch values, or attention output were discarded. If training loss falls while held-out accuracy stays at one of two, inspect the two held-out logits individually rather than increasing steps outside the protocol.

The fixed training certificate is intentionally short. Record update zero before optimization and updates one through twelve after each step. Only train examples may enter the loss used by `backward`. Evaluation may run repeatedly under `model.eval()` and `no_grad`, but must not change parameters, optimizer state, fixture arrays, or split order. The actual-feature fingerprint guards against a caller attaching trusted IDs to substituted image bytes; the aligned-target fingerprint guards against attaching a valid image to the wrong label.

Finally compare the representation choice with a CNN. Four 2x2 patches give five tokens including class, so one attention head forms 25 token-pair scores per image. Halving patch side to one produces seventeen tokens and 289 scores, more than eleven times as many. A small convolution instead shares a local kernel and keeps a spatial feature map. Neither observation decides the model automatically: state the data volume, CPU limit, required detail, and falsifying held-out comparison.

**Checkpoint 7A.** Why does the full asymmetric patch matrix catch an error that a shape assertion misses?

**Checkpoint 7B.** State the exact p17 order from positioned tokens through class logits.

**Collected answers.** A swapped middle token or ignored position can preserve shape while changing semantics; compute attention on positioned Q/K/V, add that output to the positioned tensor, apply LayerNorm, select normalized token 0, then apply the head.


In [ ]:
import torch

image = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)
patches = (image.reshape(1, 1, 2, 2, 2, 2)
                 .permute(0, 2, 4, 1, 3, 5)
                 .contiguous().reshape(1, 4, 4))
expected = torch.tensor([[[0, 1, 4, 5], [2, 3, 6, 7],
                          [8, 9, 12, 13], [10, 11, 14, 15]]], dtype=torch.float32)
assert torch.equal(patches, expected)
print(patches)


In [ ]:
import torch
from torch import nn

torch.manual_seed(20260901)
tokens = torch.arange(40, dtype=torch.float32).reshape(1, 5, 8) / 10
attention = nn.MultiheadAttention(8, 1, dropout=0, batch_first=True)
norm = nn.LayerNorm(8, eps=1e-5)
attended, _ = attention(tokens, tokens, tokens, need_weights=False)
encoded = norm(tokens + attended)
assert encoded.shape == (1, 5, 8)
print(tuple(tokens.shape), tuple(attended.shape), tuple(encoded.shape))
print(encoded.mean(-1), encoded.var(-1, unbiased=False))


## 8. Common pitfalls and Exam connections

Common pitfalls are swapping channel and token axes, flattening all patches into one token, concatenating positions as tokens, applying softmax before cross-entropy, and evaluating on train IDs. Exam connections often give a literal geometry and ask for the first wrong shape or the attention-cost ratio; write the ledger before arithmetic.

**Checkpoint 8A.** Which practice isolates patch ordering?

**Checkpoint 8B.** Which practice compares ViT and CNN constraints?

**Collected answers.** p06 isolates ordering; p21 makes the architecture choice.


## 9. Forward-only deeper topic: hierarchical vision Transformers

Forward-only deeper topic: later architectures merge patches across stages, use shifted local windows, or distill from larger models. These ideas are named for orientation only. They are not prerequisites, permitted shortcuts, or assessed APIs in B2-021; the unit uses one fixed patch scale and ordinary one-head attention.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Are shifted windows assessed here?

**Checkpoint 9B.** What attention form remains required?

**Collected answers.** No; ordinary one-head attention over the five-token sequence remains required.
